# DATA CLEANING

objective:
- merging all for datasets into one
- column standardisation
- missing values check
- converting datatypes 
- duplicate check
- handling 'EXCEPTION HANDLER' items
- datatype validation
- consistency checks
- feature creation

In [ ]:
# importing necessary files

import pandas as pd

# setting for float display
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)  # ensuring all columns are visible

In [ ]:
# loading the datasets for jan
df_jan_data = pd.read_csv("..\\data\\raw data\\pca_202601.csv")

df_jan_data.shape # 27 columns

In [ ]:
# loading the datasets for feb
df_feb_data = pd.read_csv("..\\data\\raw data\\pca_202602.csv")
df_feb_data.shape #27 columns

In [ ]:
# loading the datasets for mar
df_mar_data = pd.read_csv("..\\data\\raw data\\pca_202603.csv")
df_mar_data.shape #27 columns

In [ ]:
# loading the datasets for mar
df_apr_data = pd.read_csv("..\\data\\raw data\\pca_202604.csv")
df_apr_data.shape # 27 columns

In [ ]:
# validating the columns and datatypes for each dataset so that we can merge them together

df_jan_data.columns.equals(df_feb_data.columns) # True
df_feb_data.columns.equals(df_mar_data.columns) # True
df_mar_data.columns.equals(df_apr_data.columns) # True


# check dtypes
df_jan_data.dtypes
df_feb_data.dtypes
df_mar_data.dtypes
df_apr_data.dtypes

##### standardising columns names

In [ ]:
# standardising the column names to lower letters for all datasets
df_jan_data.columns = df_jan_data.columns.str.lower()
df_feb_data.columns = df_feb_data.columns.str.lower()
df_mar_data.columns = df_mar_data.columns.str.lower()
df_apr_data.columns = df_apr_data.columns.str.lower()

In [ ]:
# concatenating the datasets 
df_prescribing_data = pd.concat([df_jan_data, df_feb_data, df_mar_data, df_apr_data], ignore_index=True)

df_prescribing_data.shape #2252531 rows

In [ ]:
# checking year_month data
df_prescribing_data["year_month"].value_counts().sort_index()

In [ ]:
# saving the merged dataset in parquet format
df_prescribing_data.to_parquet("../data/merged data/nhs_prescribing_2026_jan_apr_merged.parquet", index=False)

### Merge Summary

- successfully combined the NHS prescribing datasets from January to April 2026 into a single analytical dataset
- This merged dataset contains 2,252,531 prescribing records across 27 rows
- This merged dataset was saved to the merged data folder to enable efficient processing and ensure a reproducible analysis workflow.

## DATA CLEANING AND VALIDATION

In [ ]:
# CHECK FOR MISSING VALUES

df_prescribing_data.isnull().sum()
# missing values are found in only two columns: snomed code and supplier name.  There is only 1 missing value in the bnf_equivalent_name column

In [ ]:
# CHECK DATATYPES

df_prescribing_data.info() # columns are either int64, float64 or str

In [ ]:
# converting identifier columns from int/float to string 

str_columns = ["year_month", "snomed_code" , "bnf_paragraph_code", "bnf_section_code", "bnf_chapter_code", "prep_class", "prescribed_prep_class" ]

for col in str_columns:
    df_prescribing_data[col] = (
            df_prescribing_data[col].astype("Int64").astype("string")
)

df_prescribing_data.info()

In [ ]:
df_prescribing_data.head(6)

In [ ]:
# CHECK FOR DUPLICATE VALUES

df_prescribing_data.duplicated().sum()
# no duplicated values found

In [ ]:
# DATA CONSISTENCY CHECK

(df_prescribing_data['nic'] <0).sum() # no negative values in NIC column
(df_prescribing_data['items'] <0).sum() # no negative values in items column
(df_prescribing_data['total_quantity'] <0).sum() # no negative values in total quantity column

In [ ]:
# RANGE CHECK

df_prescribing_data.describe()

In [ ]:
# checking which items have the largest nic
df_prescribing_data.nlargest(10, "nic")[
    ["nic", "bnf_presentation_name", "generic_bnf_equivalent_name"]
]
# they are sensors or kwikpens

In [ ]:
# exception check

exception_mask = df_prescribing_data["bnf_presentation_name"].str.contains(
    "Exception",
    case=False,
    na=False
)

exception_count = exception_mask.sum()
total_rows = len(df_prescribing_data)

print(f"{exception_count:,} exception records")
print(f"{exception_count / total_rows:.2%} of the dataset")

# total 968 records have the Exception name which is 0.04% of the total records

In [ ]:
# viewing records with exception
df_prescribing_data.loc[
    exception_mask,
    [
        "year_month",
        "bnf_presentation_name",
        "generic_bnf_equivalent_name",
        "bnf_paragraph_code",
        "bnf_chapter_code",
        "prep_class",
        "items",
        "nic"
    ]
].head(10)

In [ ]:
# looking at the total items and nic for exception records
df_prescribing_data.loc[exception_mask, ["items", "nic"]].sum()

In [ ]:
# calculating the percentage NIC of the exception records 

total_nic = df_prescribing_data["nic"].sum()

print (total_nic)


exception_nic = df_prescribing_data.loc[
    exception_mask,
    "nic"
].sum()

per_nic =round((exception_nic / total_nic) * 100, 3)

print (f"{per_nic}%" )


In [ ]:
# calculating the percentage items of the exception records 

total_items = df_prescribing_data["items"].sum()

exception_items = df_prescribing_data.loc[
    exception_mask,
    "items"
].sum()

per_items = round ((exception_items / total_items) * 100, 3)

print (f"{per_items}%" )

In [ ]:
# flagging the exception records for future analysis

df_prescribing_data["is_exception_record"] = exception_mask


#### Exception Records

A total of 968 exception records (0.04% of the dataset) were identified. All belonged to BNF Paragraph Code 190201, a predefined NHS category for non-standard or individually formulated preparations that cannot be mapped to a standard medicine presentation.

These records contributed only 0.024% of total prescription items and 0.097% of total NIC, indicating a negligible impact on the overall dataset. They were therefore retained and flagged for transparency but excluded from analyses requiring standard medicine classification.

#### CLEANING TEXT FIELDS
Removing hidden white spaces and tab characters

In [ ]:
# Remove leading/trailing whitespace and hidden tab characters
text_cols = df_prescribing_data.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df_prescribing_data[col] = (
        df_prescribing_data[col]
        .str.replace("\t", "", regex=False)
        .str.strip()
    )

### FEATURE CREATION


In [ ]:
# creating new columns for further analysis
# year column (contains year)
# month column (contains only the month)
# date column (with full date)

In [ ]:
# create year column

df_prescribing_data["year"] = df_prescribing_data["year_month"].str[:4].astype(int)

# create month column
df_prescribing_data["month"] = df_prescribing_data["year_month"].str[4:].astype(int)

# create date column
df_prescribing_data["date"] = pd.to_datetime(df_prescribing_data["year_month"], format="%Y%m")


# create a month name column
df_prescribing_data["month_name"] = df_prescribing_data["date"].dt.strftime("%b")
df_prescribing_data.head(5)

In [ ]:
#!pip install pyarrow

In [ ]:
# saving the cleaned datset as csv for compatibility
df_prescribing_data.to_csv(
    "../data/cleaned data/nhs_prescribing_2026_jan_apr.csv",
    index=False
)

In [ ]:
# saving the cleaned and finialised dataset in parquet format

df_prescribing_data.to_parquet(
    "../data/cleaned data/nhs_prescribing_2026_jan_apr.parquet",
    index=False
)

### DATA CLEANING SUMMARY

- Column names were standardised for consistency.

- Four monthly datasets were successfully merged into a single analytical dataset.

- Missing values were investigated and found to be limited to two columns (snomed code and supplier name).

- Identifier variables containing classification codes were converted from numeric formats to string types to preserve their categorical meaning and     prevent inappropriate numerical interpretation.

- Duplicate records were checked, with no duplicate found.

- Exception records were flagged and retained for transparency while being excluded from analyses requiring standard medicine classifications.
  
- Text fields were standardised by removing hidden tab characters and leading/trailing whitespace to ensure consistent grouping, filtering, and    visualisation.

- New derived columns were created to support time-based analysis, including: year, month, date, and month name.

- The cleaned dataset was stored in Parquet format to reduce storage requirements and improve loading performance. Compared with the equivalent CSV export (915 MB), the Parquet file occupied only 83 MB while preserving data types and supporting faster analytical workflows.
